# ZS601 C：B + 独立尺度上限

从同一LiDAR重新初始化。C新增 `scale_bounds`，其余B参数保持一致；可通过 `OVERRIDES={'scale_bounds':'off'}` 单独关闭。
初始化及每步更新后限制1σ半轴XY<=2h、Z<=0.1h，并清理被截断坐标的Adam动量。h为固定LiDAR参考间距。
150000步、L4、SH2、seed42、位置LR周期150000、scaling_lr0.0015；200步冒烟使用相同LR日程。
数据先拷贝到/content。每1000步固定十相机RGB/深度/法向/1σ彩色椭球；val不保存NPZ；checkpoint/PLY每50000步。
每次执行新建目录，保留B。不要对运行中的队列再次全部运行。先通过冒烟并检查图像，再运行正式单元。
此notebook是C运行入口；C结果以实际日志和验收文件为准，不预先宣称完成。


## 1. 连接L4并挂载Drive（推荐Chrome）
若点击授权后长时间无回调，停止并恢复Chrome连接；不要继续写未挂载的Drive路径。

In [ ]:
from google.colab import drive
drive.mount('/content/drive', timeout_ms=300000)


In [ ]:
from pathlib import Path
import sys,subprocess,uuid,json,shutil,zipfile,csv,os
import numpy as np
import torch
CODE_REF='4c81636bc0608add0d0401cc400fc207f6ed22dc'
REPO_URL='https://github.com/VISjudy/ZS601_3DGS.git'
DATA_ZIP=Path('/content/drive/MyDrive/LCCDataset/ZS601meetingroom/ZS601meetingroom_data.zip')
HISTORY=Path('/content/drive/MyDrive/LCCDataset/zs601_output/gaussian-splattingWithMask_v3_cff221ccfb')
VAL_SOURCE=HISTORY/'images-val10.txt'
TEST_SOURCE=HISTORY/'images_test.txt'
assert os.path.ismount('/content/drive'), 'Drive mount/authentication is required'
assert HISTORY.is_dir() and DATA_ZIP.is_file()
B_ROOT=HISTORY.parent/'zs601_B_5cc0cf5547/formal_B'
assert (B_ROOT/'completed.json').is_file()
WORK=Path('/content')/('zs601_C_' +uuid.uuid4().hex[:10]);WORK.mkdir()
RESULTS=HISTORY.parent/WORK.name
assert RESULTS.resolve().is_relative_to(HISTORY.parent.resolve())
RESULTS.mkdir()
ITERATIONS=150000
COMMON={'sh_degree':2,'position_lr_init':0.000016,'position_lr_final':0.00000016,
        'position_lr_max_steps':150000,'scaling_lr':0.0015,'seed':42,'lazy_cache':100}
VAL_ELLIPSOIDS='on'
OVERRIDES={} # Independent feature overrides; default C.
def run(args,cwd=None):
    args=list(map(str,args));print('COMMAND',args,flush=True)
    logfile=RESULTS/('command_'+uuid.uuid4().hex[:10]+'.log')
    with logfile.open('x') as log,subprocess.Popen(args,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True) as p:
        for line in p.stdout:
            log.write(line)
            if not line.startswith('Reading camera'): print(line,end='',flush=True)
        if p.wait():raise subprocess.CalledProcessError(p.returncode,args)
    print('LOG',logfile)
assert torch.cuda.is_available() and 'L4' in torch.cuda.get_device_name(0)
run(['nvidia-smi'])
print({'python':sys.version,'torch':torch.__version__,'cuda':torch.version.cuda})
run(['git','clone','--branch','v3-ab',REPO_URL,WORK/'repo'])
run(['git','checkout','--detach',CODE_REF],cwd=WORK/'repo')
CODE=WORK/'repo/gaussian-splattingWithMask_v3'


## 2. 构建CUDA扩展与21项测试

In [ ]:
assert torch.version.cuda and torch.version.cuda.split('.')[0]=='12', 'This verified build requires CUDA 12; inspect a changed runtime before proceeding'
run([sys.executable,'-m','pip','install','ninja','plyfile','laspy','scipy','pillow','opencv-python-headless','tqdm'])
run([sys.executable,'-m','pip','install','cupy-cuda12x'])
import cupy as cp
assert cp.cuda.runtime.getDeviceCount()>0 and int(cp.arange(10).sum().get())==45
os.environ['MAX_JOBS']='2'
for name in ['simple-knn','diff-gaussian-rasterization']:
    run([sys.executable,'-m','pip','install','-v','--no-build-isolation',CODE/'submodules'/name])
run([sys.executable,'-c','import torch,cupy; from simple_knn._C import distCUDA2; from diff_gaussian_rasterization import GaussianRasterizer; print(torch.cuda.get_device_name(0),cupy.__version__)'],cwd=CODE)
run([sys.executable,'-m','unittest','-v','test_geometry_v3','test_scale_bounds_v3'],cwd=CODE)


## 3. 只读分析B最终checkpoint
只在新的C结果目录保存分析JSON，B输入保持不变。

In [ ]:
run([sys.executable,'analyze_model_v3.py','--checkpoint',B_ROOT/'checkpoints/iteration_150000.pth','--output',RESULTS/'b_checkpoint_analysis.json'],cwd=CODE)


## 4. 数据拷贝至本地，保持B固定划分

In [ ]:
for p in [DATA_ZIP,VAL_SOURCE,TEST_SOURCE]:assert p.is_file(),str(p)
LOCAL_ZIP=WORK/'data.zip';shutil.copy2(DATA_ZIP,LOCAL_ZIP)
DATA=WORK/'data';DATA.mkdir()
with zipfile.ZipFile(LOCAL_ZIP) as z:
    for item in z.infolist():
        target=(DATA/item.filename).resolve()
        assert target.is_relative_to(DATA.resolve()),item.filename
        assert (item.external_attr>>16)&0o170000!=0o120000,'ZIP symlink'
    z.extractall(DATA)
POINTS=DATA/'ZS601_3cm_sample.las'
INTR=DATA/'sparse/cameras.txt';POSES=DATA/'sparse/images.txt'
VAL=WORK/'images-val10.txt';TEST=WORK/'images_test.txt';TRAIN=WORK/'images_train_v3.txt'
shutil.copy2(VAL_SOURCE,VAL);shutil.copy2(TEST_SOURCE,TEST)
for p in [POINTS,INTR,POSES]:assert p.is_file(),str(p)
run([sys.executable,'prepare_v3.py','--images_file',POSES,'--val_file',VAL,'--test_file',TEST,'--output_train',TRAIN],cwd=CODE)
for p in [VAL,TEST,TRAIN]:shutil.copy2(p,RESULTS/p.name)
print('LOCAL INPUTS',DATA,POINTS,INTR,TRAIN,VAL,TEST)


In [ ]:
def train_command(group,out,iterations,resume=None,checkpoint_interval=50000):
    cmd=[sys.executable,'train_mask_v3.py','--experiment',group,'-s',DATA,'-m',out,
         '--point_cloud',POINTS,'--train_file',TRAIN,'--val_file',VAL,'--test_file',TEST,
         '--cameras_file',INTR,'--units','scene','--iterations',iterations,
         '--val_interval',1000,
         '--checkpoint_interval',checkpoint_interval,'--val_npz','off','--val_ellipsoids',VAL_ELLIPSOIDS]
    for key,value in {**COMMON,**OVERRIDES}.items():cmd+=['--'+key,str(value)]
    if resume:cmd+=['--resume',resume]
    return cmd
def verify(out,steps):
    done=json.loads((out/'completed.json').read_text());assert done['iteration']==steps
    loss=list(csv.DictReader((out/'loss_log.csv').open()))
    assert [int(r['iteration']) for r in loss]==list(range(1,steps+1))
    assert all(np.isfinite(float(v)) for r in loss for v in r.values())
    val=list(csv.DictReader((out/'val_metrics.csv').open()))
    expected=[0]+[i for i in range(1,steps+1) if i%1000==0 or i==steps]
    assert len(val)==11*len(expected)
    for i in expected:
        d=out/'val_v3'/f'iteration_{i:06d}'
        for kind in ['rgb','normal','depth']+(['ellipsoid'] if VAL_ELLIPSOIDS=='on' else []):
            assert len(list(d.glob('*_'+kind+'.png')))==10,(d,kind)
        assert not list(d.glob('*_geometry.npz'))
    assert (out/'checkpoints'/f'iteration_{steps}.pth').is_file()
    manifest=json.loads((out/'run_manifest.json').read_text())
    baseline=json.loads((B_ROOT/'run_manifest.json').read_text())
    assert manifest['inputs']==baseline['inputs'], 'B/C input identity mismatch'
    assert manifest['val_names']==baseline['val_names']
    cfg=json.loads((out/'run_config.json').read_text())
    if cfg['scale_bounds']:
        logs=[json.loads(line) for line in (out/'geometry_log.jsonl').read_text().splitlines()]
        assert all(r['size_ratio_to_limit'][-1]<=1.000001 and r['thickness_ratio_to_limit'][-1]<=1.000001 for r in logs)
        ck=torch.load(out/'checkpoints'/f'iteration_{steps}.pth',map_location='cpu',weights_only=False)
        s=ck['model'][4].exp();h=ck['reference']['spacing']
        assert torch.isfinite(s).all()
        assert (s[:,:2]<=cfg['size_ratio']*h[:,None]*(1+1e-6)).all()
        assert (s[:,2]<=cfg['thickness_ratio']*h*(1+1e-6)).all()
    assert sorted(p.name for p in (out/'checkpoints').glob('*.pth'))==sorted(f'iteration_{i}.pth' for i in sorted(set(range(50000,steps+1,50000))|{steps}))
    print('VERIFIED',out,done,'loss rows',len(loss),'val rows',len(val))
    return {'path':str(out),**done,'loss_rows':len(loss),'val_rows':len(val)}


## 5. 200步C冒烟
检查val00/val03的RGB和1σ椭球，尺寸合格仍需检查覆盖空洞。

In [ ]:
SMOKE={}
for group in ['C']:
    out=RESULTS/('smoke_'+group);run(train_command(group,out,200),cwd=CODE)
    SMOKE[group]=verify(out,200)


In [ ]:
from IPython.display import display
from PIL import Image
for name in ['val00','val03']:
    for kind in ['rgb','ellipsoid']:
        display(Image.open(RESULTS/'smoke_C/val_v3/iteration_000200'/f'{name}_{kind}.png'))


## 6. 正式C 150000步
冒烟验收完成后执行。保持可见运行单元和完整日志。

In [ ]:
assert SMOKE['C']['iteration']==200
FORMAL={}
for group in ['C']:
    out=RESULTS/('formal_'+group);run(train_command(group,out,ITERATIONS),cwd=CODE)
    FORMAL[group]=verify(out,ITERATIONS)
with (RESULTS/'workflow_verified.json').open('x') as f:
    json.dump({'code_ref':CODE_REF,'smoke':SMOKE,'formal':FORMAL},f,indent=2)
print('C completed and verified:',RESULTS)


## 7. 完成验收后释放GPU
只在workflow_verified.json存在且正式C150000完成时释放。

In [ ]:
DISCONNECT_WHEN_VERIFIED=True
if DISCONNECT_WHEN_VERIFIED:
    assert json.loads((RESULTS/'formal_C/completed.json').read_text())['iteration']==150000
    assert (RESULTS/'workflow_verified.json').is_file()
    with (RESULTS/'runtime_release_requested.json').open('x') as f:
        json.dump({'formal_completed':150000,'results':str(RESULTS)},f)
    os.sync()
    from google.colab import runtime
    runtime.unassign()
